In [1]:
!pip install transformers datasets gradio -q

In [2]:
from google.colab import files
uploaded = files.upload()

Saving prompts.jsonl to prompts.jsonl


In [3]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="prompts.jsonl", split="train")
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 192
})


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

modelo = "datificate/gpt2-small-spanish"
tokenizer = AutoTokenizer.from_pretrained(modelo)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(modelo)
print("Modelo cargado ✓")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: datificate/gpt2-small-spanish
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo cargado ✓


In [5]:
def tokenizar(ejemplo):
    texto = ejemplo["prompt"] + " " + ejemplo["completion"]
    return tokenizer(texto, truncation=True, padding="max_length", max_length=256)

dataset_tokenizado = dataset.map(tokenizar)
print("Dataset tokenizado ✓")

Map:   0%|          | 0/192 [00:00<?, ? examples/s]

Dataset tokenizado ✓


In [6]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir="./modelo_peliculas",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=100,
    logging_steps=10,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset_tokenizado,
    data_collator=data_collator,
)

trainer.train()
print("Entrenamiento completo ✓")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.605270
20,3.731069
30,3.523857
40,3.297495
50,3.150586
60,2.804956
70,2.692061
80,2.513248
90,2.581218
100,2.411013


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Entrenamiento completo ✓


In [8]:
import gradio as gr
from transformers import pipeline

generador = pipeline("text-generation", model=model, tokenizer=tokenizer)

def recomendar(prompt):
    try:
        resultado = generador(
            prompt.strip(),
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.3,
            top_p=0.9
        )
        return resultado[0]["generated_text"]
    except Exception as e:
        return f"Error: {str(e)}"

css = """
body, .gradio-container {
    background-color: #080a0f !important;
    font-family: 'Georgia', serif !important;
}
.gradio-container {
    max-width: 800px !important;
    margin: 0 auto !important;
}
h1 {
    font-size: 3rem !important;
    font-weight: 300 !important;
    color: #c9a96e !important;
    text-align: center !important;
    letter-spacing: 0.1em !important;
    margin-bottom: 0.25rem !important;
}
.subtitle {
    color: #7a7870 !important;
    text-align: center !important;
    letter-spacing: 0.25em !important;
    font-size: 0.75rem !important;
    text-transform: uppercase !important;
}
label span {
    color: #c9a96e !important;
    font-size: 0.8rem !important;
    letter-spacing: 0.2em !important;
    text-transform: uppercase !important;
}
textarea, input[type=text] {
    background: #0e1118 !important;
    border: 1px solid rgba(201,169,110,0.25) !important;
    color: #e8e4d9 !important;
    border-radius: 0 !important;
    font-family: 'Georgia', serif !important;
    font-size: 1rem !important;
}
textarea:focus, input[type=text]:focus {
    border-color: #c9a96e !important;
    box-shadow: 0 0 0 2px rgba(201,169,110,0.1) !important;
}
button.primary {
    background: #c9a96e !important;
    color: #080a0f !important;
    border: none !important;
    border-radius: 0 !important;
    font-family: 'Georgia', serif !important;
    letter-spacing: 0.2em !important;
    text-transform: uppercase !important;
    font-size: 0.8rem !important;
}
button.primary:hover {
    background: #e8c99a !important;
}
button.secondary {
    background: transparent !important;
    border: 1px solid rgba(201,169,110,0.3) !important;
    color: #7a7870 !important;
    border-radius: 0 !important;
    letter-spacing: 0.15em !important;
}
.examples-holder {
    background: #0e1118 !important;
    border: 1px solid rgba(201,169,110,0.1) !important;
}
.example {
    color: #c9a96e !important;
    font-size: 0.82rem !important;
}
footer { display: none !important; }
"""

demo = gr.Interface(
    fn=recomendar,
    inputs=gr.Textbox(
        label="¿Cómo te sientes hoy?",
        placeholder="Ej: quiero algo de terror psicológico que no sea tan gore…",
        lines=3
    ),
    outputs=gr.Textbox(
        label="Recomendación",
        lines=6
    ),
    title="✦ CineVibeAI",
    description="<p class='subtitle'>Recomendaciones cinematográficas inteligentes</p>",
    examples=[
        ["Quiero algo para llorar"],
        ["Algo romántico pero no cursi"],
        ["Terror psicológico que no sea gore"],
        ["Quiero reírme con amigos"],
        ["Algo de ciencia ficción profunda"]
    ],
    css=css,
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.stone,
        neutral_hue=gr.themes.colors.stone,
        font=gr.themes.GoogleFont("Cormorant Garamond")
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://940ef6a17edb2b0a86.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
